#### **Step 1: Import Required Libraries**


In [24]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp, window, row_number)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql import SparkSession

StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 26, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [25]:
# Create path variable
raw_user_devices_path = "Files/cdp_bronze/postgresql/raw/user_devices"
df_cleaned_user_devices = spark.read.format("parquet").load(raw_user_devices_path)
display(df_cleaned_user_devices.limit(10))

StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 27, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 764e9888-ea9d-4069-bccc-e60f97f38c94)

### **Column-by-Column Transformation Plan**
device_id
- Current: Already in consistent custom format (e.g., fee18459-ab07-47c1-b842-d76efafb9b61).
- Action: No transformation needed.
- Final: Keep as-is.

user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

#### **Step 3: Inspect the user_devices dataframe for data quality checks**

In [26]:
# Select specific columns and count nulls
null_counts = df_cleaned_user_devices.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['device_id','user_id','device_type', 'ip_address', 'last_used']
    ]
)

# Display results
null_counts.show()

StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 28, Finished, Available, Finished)

+---------+-------+-----------+----------+---------+
|device_id|user_id|device_type|ip_address|last_used|
+---------+-------+-----------+----------+---------+
|        0|      0|          0|         0|        0|
+---------+-------+-----------+----------+---------+



#### **Step 4: Data Transformation to device_type column**

Issues:
- Inconsistent casing (e.g., macbook, iPhone, Smart TV, etc.)
- Non-standard or humorous entries (e.g., SmartToaster)
- Granular/raw values not suitable for high-level analytics (e.g., MacBook, iPhone, Linux)
- Unrecognized or blank entries may exist (empty string or NULL)

Transformations:
- Normalize casing: Convert all values to title case (e.g., macbook → MacBook)
- Map raw types to standard categories:
- MacBook, Windows PC, Linux → 'Desktop/Laptop'
- iPhone, Android → 'Mobile'
- Tablet → 'Tablet'
- Smart TV → 'Smart Device'
- SmartToaster → 'IoT Device'
- Handle unknown or empty values:
- Replace empty strings or unrecognized values with NULL or 'Unknown' (preferably NULL for clean analytics)


In [27]:
# Check distinct values in device_type
# df_cleaned_user_devices.select("device_type").distinct().show()

# Normalize and transform device_type values
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "device_type",
    when(lower(trim(col("device_type"))).isin("macbook", "windows", "linux"), "Desktop/Laptop")
    .when(lower(trim(col("device_type"))).isin("iphone", "android"), "Mobile")
    .when(lower(trim(col("device_type"))) == "tablet", "Tablet")
    .when(lower(trim(col("device_type"))) == "smart tv", "Smart Device")
    .when(lower(trim(col("device_type"))) == "smarttoaster", "IoT Device")
    .otherwise(None)  # Set anything else to NULL
)

# Display distinct values after transformation
df_cleaned_user_devices.select("device_type").distinct().show()


StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 29, Finished, Available, Finished)

+--------------+
|   device_type|
+--------------+
|  Smart Device|
|    IoT Device|
|        Mobile|
|        Tablet|
|Desktop/Laptop|
|          NULL|
+--------------+



In [28]:
# Check distinct values in ip_address
# df_cleaned_user_devices.select("ip_address").distinct().show()

# Step 1: Define a basic IPv4 regex pattern (match valid dotted quad)
ipv4_regex = r'^((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}$'

# Step 2: Clean and standardize the 'ip_address' column
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "ip_address",
    when(
        trim(col("ip_address")).rlike(ipv4_regex),  # Matches valid IPs
        trim(col("ip_address"))
    ).otherwise(None)  # Set malformed/invalid to NULL
)

# Optional: Keep only valid entries (if you want to drop bad rows)
# df_cleaned_user_devices = df_cleaned_user_devices.filter(col("ip_address").isNotNull())

# Display distinct cleaned IPs
df_cleaned_user_devices.select("ip_address").distinct().show(truncate=False)


StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 30, Finished, Available, Finished)

+---------------+
|ip_address     |
+---------------+
|212.232.247.39 |
|100.161.158.42 |
|75.46.80.127   |
|197.209.152.121|
|199.154.54.219 |
|26.39.243.161  |
|192.156.33.93  |
|119.94.215.108 |
|17.230.66.102  |
|161.102.205.211|
|156.150.121.177|
|181.246.225.181|
|102.54.40.91   |
|15.103.26.107  |
|105.57.180.193 |
|60.13.150.232  |
|212.177.157.54 |
|223.4.86.115   |
|145.218.147.199|
|14.48.101.244  |
+---------------+
only showing top 20 rows



#### **Step 5: Data Transformation to last_used column**
Issues:
- Multiple inconsistent formats (6+ variations)
- Could lead to parsing errors during transformation

Transformations:
- Parse string to standardized TIMESTAMP format
- Use multi-format parsing (PySpark or Pandas with try/except)
- Convert all to ISO format or UTC timezone if needed

In [29]:
from pyspark.sql.functions import (
    col, to_date, coalesce, regexp_replace, date_format
)

# Preview original distinct values
# print("Original last_used values:")
# df_cleaned_user_devices.select("last_used").distinct().show(truncate=False)

# STEP 1: Parse common known date formats
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "last_used_parsed",
    coalesce(
        to_date("last_used", "MMM dd, yyyy HH:mm"),      # Mar 24, 2025 21:22
        to_date("last_used", "dd MMMM yyyy, HH:mm"),     # 25 February 2025, 18:45
        to_date("last_used", "dd/MM/yyyy HH:mm"),        # 28/09/2024 02:54
        to_date("last_used", "yyyy-MM-dd HH:mm:ss"),     # 2024-07-28 12:42:13
        to_date("last_used", "MM-dd-yyyy hh:mm a"),      # 06-25-2024 05:00 PM
        to_date("last_used", "yyyy/MM/dd HH:mm:ss"),     # 2024/10/24 15:24:00
        to_date("last_used", "MMM dd, yyyy"),            # Oct 01, 2024
        to_date("last_used", "yyyy-MM-dd"),              # 2024-10-24
        to_date("last_used", "MM/dd/yyyy")               # 07/14/2024
    )
)

# STEP 2: Fallback to regex cleanup for less standard formats
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "last_used_parsed",
    coalesce(
        col("last_used_parsed"),
        to_date(regexp_replace(col("last_used"), r"([0-9]{2}) ([A-Za-z]{3}) ([0-9]{4})", "$2 $1, $3")),
        to_date(regexp_replace(col("last_used"), r"([0-9]{2})-([0-9]{2})-([0-9]{4})", "$2/$1/$3"))
    )
)

# STEP 3: Format final result and overwrite original last_used column
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "last_used",
    date_format(col("last_used_parsed"), "yyyy-MM-dd")
)

# STEP 4: Drop intermediate column
df_cleaned_user_devices = df_cleaned_user_devices.drop("last_used_parsed")

# Preview final distinct values
print("Standardized last_used values:")
df_cleaned_user_devices.select("last_used").distinct().show(truncate=False)

# Show sample output
display(df_cleaned_user_devices.limit(5))


StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 31, Finished, Available, Finished)

Standardized last_used values:
+----------+
|last_used |
+----------+
|2024-09-15|
|2024-08-20|
|2023-05-18|
|2024-10-24|
|2024-01-19|
|2024-07-14|
|2024-10-22|
|2024-08-06|
|2025-02-25|
|2024-04-03|
|2024-12-12|
|2025-03-20|
|2023-06-07|
|2025-04-05|
|2024-02-28|
|2024-02-08|
|2024-07-12|
|2025-01-03|
|2023-07-21|
|2023-10-26|
+----------+
only showing top 20 rows



SynapseWidget(Synapse.DataFrame, 754110da-67f3-4de1-a4c8-3232baf248b0)

#### **Step 6: Data Deduplication on `user_id` Column in Device Table**

**Issue:**
- Multiple entries for the same `user_id` were found in the device records, due to users switching devices or reconnecting.

**Transformation:**
- Used a window function partitioned by `user_id` and ordered by `last_used` (most recent first).
- Retained only the latest record per user by assigning row numbers and filtering where `row_num == 1`.

**Outcome:**
- Each user is now represented by their most recently used device.

In [30]:
# Define window partitioned by user_id and ordered by last_used descending
window_spec = Window.partitionBy("user_id").orderBy(col("last_used").desc())

# Assign row numbers
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "row_num", row_number().over(window_spec)
).filter(col("row_num") == 1).drop("row_num")

# Show result
df_cleaned_user_devices.show(10, truncate=False)

StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 32, Finished, Available, Finished)

+------------------------------------+----------+--------------+---------------+----------+
|device_id                           |user_id   |device_type   |ip_address     |last_used |
+------------------------------------+----------+--------------+---------------+----------+
|71a4ff4f-7f9e-4051-83aa-ff0199244815|BB10000004|Smart Device  |77.254.42.34   |NULL      |
|6519e063-8ef1-4bf3-a2b2-1903d333889e|BB10000006|Mobile        |145.205.224.100|2024-12-18|
|fbcea6dc-3db2-495c-8158-7c9be499ae28|BB10000010|Desktop/Laptop|19.59.103.90   |2024-02-09|
|3489f797-0abe-4379-a01b-eb9fefe4b41f|BB10000014|Desktop/Laptop|66.13.238.40   |2024-12-15|
|c6c7553f-25ce-41f6-bf27-a0469ea2f3ea|BB10000019|Mobile        |49.19.169.242  |2024-11-18|
|6df52cb5-c0d0-4ba4-ac74-55e62804aa57|BB10000020|Tablet        |14.27.107.183  |2025-04-16|
|dc2a7d70-a9ea-4f10-9f6d-bddcc421af02|BB10000023|Mobile        |0.0.0.0        |2025-01-03|
|4c80a75f-2743-4e87-b0b1-ba4b9a2ad3ad|BB10000026|Smart Device  |221.138.88.147 |

#### **Step 7: Data Load to Silver Lakehouse Layer**
This script processes cleaned transaction data and writes it to the Silver layer in Delta format. It performs incremental loading using a MERGE strategy to avoid duplicates and ensure upserts (insert new records only).

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/postgresql/cleaned/cleaned_user_devices.

Prepare Columns for Consistency:
- last_used: Ensures a proper date column is available for partitioning.
- load_timestamp: Adds a unique timestamp (once per job) for tracking the data load time.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key device_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by last_used for efficient querying and file organization.

In [31]:
# === Spark Session ===
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# === Define Paths ===
silver_path = "Files/cdp_silver/postgresql/cleaned/cleaned_user_devices"
temp_path = "/tmp/delta_temp_user_devices"  # Temporary staging path

# === Prepare DataFrame ===
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "last_used", to_date("last_used")
)

load_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_cleaned_user_devices = df_cleaned_user_devices.withColumn(
    "load_timestamp", lit(load_ts)
)

# === Write & Merge Logic ===
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    # Optional staging to avoid partition path conflict
    df_cleaned_user_devices.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .option("tempPath", temp_path) \
        .partitionBy("last_used") \
        .save(silver_path)

    delta_table.alias("target").merge(
        df_cleaned_user_devices.alias("source"),
        "target.device_id = source.device_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df_cleaned_user_devices.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("last_used") \
        .save(silver_path)

print("✅ Cleaned user devices loaded to Silver layer.")


StatementMeta(, 49bd662a-329c-4cf8-8a22-32ef99826a73, 33, Finished, Available, Finished)

✅ Cleaned user devices loaded to Silver layer.
